# M11.3 — Export M10 Step 3 / 10_2 (GAP hierarchical) → Hub model clone

Plan: [`plans/milestone_11/11_huggingface_artifacts_plan.md`](../../plans/milestone_11/11_huggingface_artifacts_plan.md).  
Architecture: [`plans/00_architecture.md`](../../plans/00_architecture.md) §5.7.

Extracts the **10_2 pooled hierarchical (GAP) head only** (`model_pooled`) from
`checkpoints/m10/m10_hierarchical_light_then_camera.pt` into the local clone of
[`tbhugging/gummybear_hierarchical_fusion`](https://huggingface.co/tbhugging/gummybear_hierarchical_fusion).

This is **not** the Fourier 10_2 variant stored in the same study checkpoint.

Machine-local staging path: gitignored `configs/hf/local.toml`
(copy from `configs/hf/local.toml.example`). This notebook writes weights + card
into that clone; it does **not** push to the Hub.

In [ ]:
from pathlib import Path
import shutil
import sys
import subprocess

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

for name in ("build", "dist"):
    shutil.rmtree(ROOT / name, ignore_errors=True)
for egg in (ROOT / "src").glob("*.egg-info"):
    shutil.rmtree(egg, ignore_errors=True)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-cache-dir",
        f"{ROOT}[dl,hf,dev]",
        "-c",
        str(ROOT / "requirements.txt"),
    ]
)

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")

## Resolve local clone + export

Requires:
- `configs/hf/local.toml` with `models.gummybear_hierarchical_fusion.local_clone`
- Study checkpoint `checkpoints/m10/m10_hierarchical_light_then_camera.pt`

In [ ]:
from IPython.display import Markdown, display

from tomography_ml_huggingface import (
    export_gummybear_hierarchical_fusion,
    resolve_gummybear_hierarchical_fusion_paths,
)

paths = resolve_gummybear_hierarchical_fusion_paths(ROOT)
display(Markdown(
    f"**Hub:** [`{paths.hub_id}`]({paths.hub_url})  \n"
    f"**Local clone:** `{paths.local_clone}`"
))

result = export_gummybear_hierarchical_fusion(ROOT)
display(Markdown(
    f"Wrote `{display_path(result.weights_path)}`, "
    f"`{display_path(result.config_path)}`, "
    f"`{display_path(result.readme_path)}`  \n"
    f"n_params={result.n_params}  lr={result.lr:g}  "
    f"metrics={{{', '.join(f'{k}={v:.4f}' for k, v in sorted(result.metrics.items()))}}}"
))

## Smoke-check reload

In [ ]:
import json
import torch

from tomography_ml.localization.localize_multiview import (
    HierarchicalLightThenCameraFusionLocalizer,
)

cfg = json.loads(result.config_path.read_text(encoding="utf-8"))
assert cfg["protocol"] == "10_2"
assert cfg["backbone_kind"] == "pooled"

model = HierarchicalLightThenCameraFusionLocalizer.for_10_2_pooled(
    n_cameras=cfg["n_cameras"],
    n_lights=cfg["n_lights"],
    camera_angles_deg=cfg["camera_angles_deg"],
    light_angles_deg=cfg["light_angles_deg"],
    flat_layout=cfg["flat_layout"],
)
views = torch.zeros(1, cfg["n_lights"], cfg["n_cameras"], 1, cfg["image_height"], cfg["image_width"])
model(views)
state = torch.load(result.weights_path, map_location="cpu", weights_only=True)
model.load_state_dict(state, strict=True)
model.eval()
xyz = model(views)
print("reload ok", tuple(xyz.shape), "config n_params=", cfg["n_params"])

## Hub upload (manual)

After reviewing the local clone:

```bash
hf auth login
cd "$local_clone"   # from configs/hf/local.toml
hf upload tbhugging/gummybear_hierarchical_fusion . .
# or: git add -A && git commit && git push
```